# Sentinel IDS on Google Colab

This notebook clones the public [Sentinel-IDS GitHub repository](https://github.com/braraks024-cell/Sentinel-IDS), installs its app dependencies, and starts the Flask application in this Colab runtime. No local virtual environment, local database, or NSL-KDD training dataset is uploaded. The trained Random Forest artifacts in the repository are used for predictions.

In [ ]:
from pathlib import Path
import os
import subprocess

repo_dir = Path('/content/Sentinel-IDS')
if repo_dir.exists():
    subprocess.check_call(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'])
else:
    subprocess.check_call([
        'git', 'clone', '--depth', '1',
        'https://github.com/braraks024-cell/Sentinel-IDS.git', str(repo_dir)
    ])
os.chdir(repo_dir)
print(f'Project ready at {repo_dir}')

In [ ]:
subprocess.check_call([
    'python', '-m', 'pip', 'install', '--quiet', '-r', 'requirements.txt'
])
print('Runtime dependencies installed.')

In [ ]:
required_artifacts = [
    'saved_models/random_forest.joblib',
    'saved_models/preprocessor.joblib',
    'saved_models/label_encoder.joblib',
    'saved_models/feature_columns.json',
    'saved_models/metrics.json',
]
missing_artifacts = [name for name in required_artifacts if not (repo_dir / name).is_file()]
if missing_artifacts:
    raise FileNotFoundError('Required trained model files are missing: ' + ', '.join(missing_artifacts))
print('Trained prediction artifacts verified.')

In [ ]:
import json
import time
import urllib.error
import urllib.request
import secrets
from IPython.display import HTML, display
from google.colab.output import eval_js

os.environ.setdefault('IDS_SECRET_KEY', secrets.token_urlsafe(48))
log_path = Path('/content/sentinel-ids-app.log')
log_file = log_path.open('w')
app_process = subprocess.Popen(
    ['python', 'app.py'],
    cwd=repo_dir,
    env={**os.environ, 'HOST': '0.0.0.0', 'PORT': '5000'},
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

health_url = 'http://127.0.0.1:5000/health'
for attempt in range(60):
    if app_process.poll() is not None:
        log_file.flush()
        raise RuntimeError('Flask exited during startup. Log output:\n' + log_path.read_text())
    try:
        with urllib.request.urlopen(health_url, timeout=2) as response:
            health = json.loads(response.read().decode('utf-8'))
            if health.get('status') == 'ok':
                break
    except (urllib.error.URLError, TimeoutError):
        time.sleep(1)
else:
    log_file.flush()
    raise TimeoutError('Flask did not become healthy within 60 seconds. Log output:\n' + log_path.read_text())

proxy_url = eval_js('google.colab.kernel.proxyPort(5000)')
display(HTML(f'<h3>Sentinel IDS is ready</h3><a href="{proxy_url}" target="_blank">Open Sentinel IDS</a>'))
print('Health check:', health)

## Using the app

Open the **Open Sentinel IDS** link from the preceding cell, then register and sign in. Keep this notebook runtime active while using the app. Colab runtimes are temporary; the SQLite database and uploaded analysis data are not durable and may be erased when the runtime ends. Do not upload sensitive or private network data to this educational demo.